# Chapter 1 &mdash; Recursively Enumerable: Patterns Defined by What Programs DO

**Concept 14 of the Chapter 1 decomposition:** *Pattern Class IV — Recursively Enumerable (Turing-Recognizable) Patterns*

Adding the words "<b>such that it halts</b>" to a set of programs jacks the pattern class all the way up &mdash; and makes "no" undiscoverable.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Recursively-Enumerable-Patterns/Concept-Recursively-Enumerable-Patterns.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Compare two descriptions:

* the set of all $(M,w)$ pairs where $M$ is a program and $w$ an input &mdash; mere *syntax*;
* the same set **such that** $M$ run on $w$ does not loop &mdash; about *behaviour*.

The "such that" clause ties membership to what happens when you **execute**, and that
dramatically enriches the class. Hence the other name: **Turing-recognisable**.

The asymmetry to take away: **halting is discoverable, looping is not.**

## 2. Definitions

### A collection of small programs, some of which loop

In [ ]:
def p_halts_fast(n):  return n + 1
def p_halts_slow(n):
    t = 0
    for _ in range(n * 500): t += 1
    return t
def p_loops(n):
    while True:                 # never returns
        pass

PROGRAMS = {'halts_fast': p_halts_fast,
            'halts_slow': p_halts_slow,
            'loops'     : p_loops}

### Dovetailing: run everything a little at a time

You may **not** run one program to completion before starting the next &mdash; if it loops
you would never get back. So interleave: one step of each, round-robin.

This is the construction behind "recursively **enumerable**".

In [ ]:
def stepper(fn, n, budget):
    """Pretend to run fn(n) for `budget` steps. Returns (done, value)."""
    if fn is p_loops:
        return (False, None)                 # never finishes, whatever the budget
    if fn is p_halts_slow and budget < 40:
        return (False, None)                 # needs more steps
    return (True, fn(n))

def dovetail(programs, inputs, rounds=60):
    """Round-robin over all (program, input) pairs, growing the budget.
       Prints each pair as it is discovered to halt."""
    found = []
    for r in range(1, rounds + 1):
        for name in list(programs)[:r]:      # reveal one more program each round
            for w in inputs[:r]:
                done, val = stepper(programs[name], w, r)
                if done and (name, w) not in found:
                    found.append((name, w))
                    print("  round %2d : %-11s on %s HALTS -> %s" % (r, name, w, val))
    return found

<!-- nav-strip -->

---

&larr;&nbsp;[Ch1&nbsp;13.&nbsp;Pattern Class III — Context-Sensitive Patterns](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Context-Sensitive-Patterns/Concept-Context-Sensitive-Patterns.ipynb) &nbsp;&middot;&nbsp; [**Chapter 1** index](https://github.com/ganeshutah/Jove/blob/master/Chapter1/README.md) &nbsp;&middot;&nbsp; [Ch1&nbsp;15.&nbsp;FA, PDA, and LBA as Simplified Turing Machines](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-FA-PDA-LBA-As-Simplified-TMs/Concept-FA-PDA-LBA-As-Simplified-TMs.ipynb)&nbsp;&rarr;

---

## 3. Tests

Dovetailing lists every halting pair **eventually**. Nothing is starved by the looper.

In [ ]:
found = dovetail(PROGRAMS, [1, 2, 3], rounds=50)
print()
print("listed", len(found), "halting pairs")
assert found and all(name != "loops" for name, _ in found)
print("note  : 'loops' never appears -- and we never had to decide that it wouldn't")

The asymmetry, stated plainly.

In [ ]:
print("If M halts on w : we find out, by waiting. ('yes' is discoverable)")
print("If M loops on w : we wait forever.         ('no'  is NOT discoverable)")
print()
print("So the set of halting pairs is RECURSIVELY ENUMERABLE (listable)")
print("but NOT RECURSIVE (decidable). Chapter 14 gives these names properly.")

Why running to completion would be fatal.

In [ ]:
order = ['loops', 'halts_fast']
print("naive order:", order)
print("running 'loops' to completion first means 'halts_fast' is NEVER reached.")
print("Fair, round-robin scheduling is what makes the enumeration work.")

## 4. Exercises


1. Add a program that halts only on even inputs. Does dovetailing still list all its
   halting pairs?
2. `stepper` cheats: it *knows* `p_loops` never finishes. Why can no real
   implementation know that? (This is the halting problem, Chapter 15.)
3. Sorting can be seen as the set of pairs (input array, sorted output). Why is that
   a much tamer recursively enumerable set than the halting one?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter1/Concept-Recursively-Enumerable-Patterns')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')